# 05 - Clustering Ulasan dengan K-Means

**Tahap 3 dan 4** metodologi: representasi teks dengan TF-IDF, lalu pengelompokan
ulasan menggunakan K-Means. Jumlah cluster optimal ditentukan dengan
**Elbow Method** dan **Silhouette Score**.

Clustering di sini bertujuan menemukan **tema ulasan secara otomatis**, tanpa
mengandalkan daftar aspek buatan manusia. Hasilnya nanti dibandingkan dengan
analisis aspek pada notebook 04 — bila keduanya sejalan, kesimpulan menjadi
lebih kuat.

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


## 5.1 Memuat data hasil preprocessing

In [2]:
from src import io_utils, features

df = io_utils.load_csv(config.PREP_CSV)
docs = df["tokens_stemmed"].fillna("").astype(str)
print(f"Ulasan: {len(df):,}")
print(f"Ulasan tanpa token: {int(docs.str.strip().eq('').sum()):,}")

Ulasan: 6,795
Ulasan tanpa token: 161


## 5.2 Representasi TF-IDF

Parameter yang dipakai:

| Parameter | Nilai | Alasan |
|---|---|---|
| `ngram_range` | (1, 2) | menangkap frasa seperti "live musik", "harga terjangkau" |
| `min_df` | 2 | membuang kata yang hanya muncul sekali (noise) |
| `max_df` | 0.95 | membuang kata yang muncul di hampir semua ulasan |
| `sublinear_tf` | True | meredam dominasi kata yang sangat sering |

In [3]:
vectorizer, matrix, tfidf_summary = features.run_tfidf(docs)
io_utils.save_csv(tfidf_summary, config.TABLES_DIR / "05_ringkasan_tfidf.csv")
io_utils.save_sparse(matrix, config.TFIDF_MATRIX_NPZ)
io_utils.save_pickle(vectorizer, config.TFIDF_VECTORIZER_PKL)
tfidf_summary

,Metrik,Nilai
0,Jumlah dokumen,6795.0000
1,Jumlah fitur (term),6885.0000
2,Total nilai non-zero,66454.0000
3,Kepadatan matriks (%),0.1420
4,Nilai minimum non-zero,0.0235
5,Nilai maksimum,1.0000
6,rata-rata fitur per dokumen,9.7800


In [4]:
features.term_document_frequency(vectorizer, matrix).head(20).to_frame("Jumlah dokumen")

,Jumlah dokumen
musik,1435
nongkrong,1430
makan,1304
live,1290
live musik,1202
enak,1153
harga,907
nyaman,847
layan,717
menu,646


## 5.3 Penentuan jumlah cluster optimal

Untuk setiap k dari 2 sampai 10 dihitung:

- **Inertia** — untuk Elbow Method (semakin kecil semakin rapat).
- **Silhouette Score** — seberapa baik objek berada di clusternya (semakin
  besar semakin baik, maksimum 1).
- **Davies-Bouldin Index** — rasio sebaran dalam cluster terhadap jarak antar
  cluster (semakin kecil semakin baik).

In [5]:
from src import clustering

evaluation, models = clustering.evaluate_k(matrix)
io_utils.save_csv(evaluation, config.K_EVAL_CSV)
evaluation

,k,Inertia,Silhouette,Davies-Bouldin,Ukuran Terkecil,Ukuran Terbesar,Penurunan Inertia
0,2,6352.99,0.0188,5.7074,936,5859,0.00
1,3,6286.30,0.0209,5.4482,498,5341,-66.69
2,4,6247.18,0.0203,6.0017,355,4882,-39.12
3,5,6206.48,0.0202,5.4727,180,4802,-40.70
4,6,6187.92,0.0199,7.4904,242,3527,-18.56
5,7,6172.19,0.0215,5.5806,92,4398,-15.73
6,8,6120.84,0.0182,5.7322,73,3827,-51.35
7,9,6103.57,0.0243,5.5183,155,3809,-17.27
8,10,6057.76,0.0215,5.1667,63,3702,-45.81


In [6]:
from src import visualize

knee = clustering.elbow_knee(evaluation)
best_k = int(evaluation.loc[evaluation["Silhouette"].idxmax(), "k"])

visualize.plot_elbow(evaluation, knee)
visualize.plot_silhouette(evaluation, best_k)
visualize.plot_dbi(evaluation)
print(f"k terbaik menurut Silhouette : {best_k}")
print(f"k siku menurut Elbow         : {knee}")

k terbaik menurut Silhouette : 9
k siku menurut Elbow         : 5


## 5.4 Memilih k dan menjalankan K-Means final

Aturan pemilihan: **k dengan Silhouette tertinggi**, dengan Elbow sebagai
pendukung. Jika hasil kedua metode berbeda, Silhouette dipakai karena
memberi ukuran kualitas pemisahan yang eksplisit.

In [7]:
from src import evaluate

result = clustering.run_clustering(matrix, vectorizer)
chosen_k = result["chosen_k"]

print(f"k dipakai: {chosen_k}")
print(f"Silhouette final: {result['silhouette_final']}  -> {evaluate.interpret_silhouette(result['silhouette_final'])}")
print(f"Davies-Bouldin final: {result['dbi_final']}")

k dipakai: 4
Silhouette final: 0.0203  -> Struktur cluster lemah (umum pada teks ulasan pendek)
Davies-Bouldin final: 6.0017


In [8]:
metrics = evaluate.clustering_metrics_table(result)
io_utils.save_csv(metrics, config.TABLES_DIR / "05_metrik_cluster.csv")
metrics

,Metrik,Nilai
0,Jumlah cluster terpilih (k),4
1,Silhouette Score (final),0.0203
2,Davies-Bouldin Index (final),6.0017
3,k terbaik menurut silhouette,4
4,k terbaik menurut elbow,5
5,Alasan pemilihan k,"Silhouette tertinggi pada k=4 sebesar 0.0203 (Davies-Bouldin 6.0017, cluster terkecil ..."


In [9]:
result["sizes"]

,Cluster,Jumlah,Persentase
0,0,912,13.42
1,1,646,9.51
2,2,4882,71.85
3,3,355,5.22


In [10]:
visualize.plot_cluster_sizes(result["sizes"])

WindowsPath('D:/hikam/tugas DataMining/results/figures/05_ukuran_cluster.png')

In [11]:
result["per_cluster"]

,Cluster,Ukuran,Silhouette Rata-rata,Silhouette Minimum,Silhouette Maksimum
0,0,912,0.0405,-0.0218,0.1913
1,1,646,0.0168,-0.0386,0.0995
2,2,4882,0.0103,-0.0404,0.0518
3,3,355,0.1126,0.0378,0.2891


## 5.5 Term dominan per cluster

Term dengan rata-rata bobot TF-IDF tertinggi menunjukkan tema tiap cluster.

In [12]:
result["top_terms"].pivot_table(
    index="Term", columns="Cluster", values="Bobot TF-IDF", fill_value=0
).round(3)

Cluster,0,1,2,3
Term,,,,
asik,0.020,0.000,0.016,0.080
asik nongkrong,0.000,0.000,0.000,0.081
asyik,0.000,0.000,0.000,0.028
bagus,0.019,0.000,0.020,0.000
bareng,0.000,0.000,0.000,0.019
cocok,0.015,0.000,0.000,0.031
cocok nongkrong,0.000,0.000,0.000,0.040
cozy,0.000,0.000,0.016,0.000
enak,0.023,0.232,0.000,0.078


In [13]:
for cluster, terms in result["top_terms_str"].items():
    print(f"Cluster {cluster}: {terms}")

Cluster 0: live musik, musik, live, nongkrong, nongkrong live, makan, enak, asik
Cluster 1: enak, makan, makan enak, nyaman, kopi enak, kopi, harga, minum
Cluster 2: nyaman, layan, makan, bagus, harga, kopi, nongkrong, ramai
Cluster 3: nongkrong, enak nongkrong, asik nongkrong, asik, enak, cocok nongkrong, cocok, nyaman nongkrong


In [14]:
visualize.plot_cluster_heatmap(result["top_terms"])

WindowsPath('D:/hikam/tugas DataMining/results/figures/05_heatmap_term.png')

## 5.6 Profil cluster

Profil menggabungkan ukuran cluster, sebaran rating, dan komposisi sentimen.

In [15]:
from src import interpret

scored = io_utils.load_csv(config.DOC_SENTIMENT_CSV)
profile = interpret.cluster_profile(scored, result["labels"], result["top_terms_str"])
io_utils.save_csv(profile, config.CLUSTER_SUMMARY_CSV)
io_utils.save_csv(result["sizes"], config.TABLES_DIR / "05_ukuran_cluster.csv")
profile

,Cluster,Jumlah Ulasan,Persentase,Rating Rata-rata,Rating 1-2,Rating 3,Rating 4-5,Tema (otomatis),Term Dominan,Sentimen Positif (%),Sentimen Negatif (%),Sentimen Netral (%),Sentimen Dominan
0,2,4882,71.85,4.126,570,529,3783,Suasana (beragam),"nyaman, layan, makan, bagus, harga, kopi, nongkrong, ramai",58.77,17.53,23.70,Positif
1,0,912,13.42,4.634,11,47,854,Live Music & Suasana,"live musik, musik, live, nongkrong, nongkrong live, makan, enak, asik",97.59,1.97,0.44,Positif
2,1,646,9.51,4.432,39,42,565,Makanan (beragam),"enak, makan, makan enak, nyaman, kopi enak, kopi, harga, minum",94.27,4.64,1.08,Positif
3,3,355,5.22,4.679,1,17,337,Suasana,"nongkrong, enak nongkrong, asik nongkrong, asik, enak, cocok nongkrong, cocok, nyaman ...",68.17,1.69,30.14,Positif


In [16]:
visualize.plot_cluster_sentiment(profile)

WindowsPath('D:/hikam/tugas DataMining/results/figures/05_sentimen_cluster.png')

## 5.7 Contoh ulasan per cluster

In [17]:
examples = interpret.cluster_examples(scored, result["labels"], n=3)
io_utils.save_csv(examples, config.TABLES_DIR / "05_contoh_cluster.csv")

for cluster in sorted(examples["Cluster"].unique()):
    print(f"\n=== Cluster {cluster} ===")
    for _, row in examples[examples["Cluster"] == cluster].iterrows():
        print(f"  [{row['Jenis']} | {row['Rating']}★] {row['Review'][:160]}")


=== Cluster 0 ===
  [Positif | 5★] Ramai banget kalo setiap datang kesini, tapi tetap dicariin kursi kok sama pelayannya. Makanan dan minumannya enak-enaak, harga juga masih tergolong affordable.
  [Positif | 5★] Tempat nkngkrong 24 jam paling ok di Jogja... kopi dan makanan nya enak ... dan ada Live Music nya kalo malam ..
  [Negatif | 1★] Menu sangat okey, tapi suasana agak gaduh, mungkin saran klo live musik tempatnya agak tertutup aja, demi kenyamanan lingkungan bersama.

=== Cluster 1 ===
  [Positif | 5★] Tempatnya super cozy. Makanannya enak, layanannya cepat dan ramah. Jelas bakal balik lagi kesini. 👍👍👍👍👍
  [Positif | 5★] Menu makanan nya enak semua<br>Layanan nya sat set banget<br>Suasana nya sangat nyaman ada live music nya bikin makin betah
  [Negatif | 1★] Baru masuk, tukang parkir (si gondrong) ud pasang muka jutek kesel gara2 masuknya lewat pintu keluar, ga ada sign nya jg, trus di kasih teguran, ngatur parkirny

=== Cluster 2 ===
  [Positif | 5★] Salah satu tempat nongkr

## 5.8 Menyimpan hasil clustering

In [18]:
clustered = scored.copy()
clustered["cluster"] = result["labels"]
io_utils.save_csv(clustered, config.CLUSTERED_CSV)
print(f"Tersimpan: {config.CLUSTERED_CSV.name} ({len(clustered):,} baris)")
clustered[["review_id", "Rating", "cluster", "sentimen_teks"]].head()

Tersimpan: 05_reviews_clustered.csv (6,795 baris)


,review_id,Rating,cluster,sentimen_teks
0,1,5,0,Positif
1,2,1,2,Negatif
2,3,3,2,Positif
3,4,1,2,Negatif
4,5,2,2,Netral


## 5.9 Kesimpulan Tahap 5

- Teks ulasan berhasil direpresentasikan sebagai matriks TF-IDF.
- Jumlah cluster dipilih berdasarkan Silhouette Score, didukung Elbow Method.
- Nilai Silhouette pada teks ulasan pendek umumnya rendah (di bawah 0,1)
  karena banyak ulasan hanya berisi 5-15 kata dan berbagi kosakata yang mirip.
  Nilai tersebut tetap bermakna untuk analisis eksploratif, tetapi harus
  disampaikan apa adanya di laporan.
- Hasil clustering dipakai pada notebook 06 bersama analisis aspek untuk
  menyusun rekomendasi.